# Sensitivity Analysis

Analisi di sensitività degli esponenti di scaling ζ(q) rispetto a perturbazioni dei pick P/S.

## 1. Imports and setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import logging
from pathlib import Path
import pickle

from src import (
    set_plot_style,
    analyze_all_windows,
    segment_all_signals,
    run_sensitivity_analysis,
    create_summary
)

colors, colors1 = set_plot_style()
logging.basicConfig(level=logging.WARNING)

print("Environment ready")

Environment ready


## 2. Configuration

In [2]:
# Dataset
DATASET = 'current'
PICKING_METHOD = 'phasenet'

# ============================================================================
# CHOOSE DATA TYPE (change this and re-run notebook for each type)
# ============================================================================
DATA_TYPE = 'acceleration'  # OPTIONS: 'acceleration', 'velocity', 'displacement'
# ============================================================================

# Coda methods to analyze
CODA_METHODS = ['rautian', 'arias', 'envelope', 'median']

# Perturbation scenarios
PERTURBATION_SCENARIOS = {
    'noise_small': {'type': 'gaussian', 'std': 0.2},
    'noise_medium': {'type': 'gaussian', 'std': 0.5},
    'noise_large': {'type': 'gaussian', 'std': 1.0},
    'bias_early': {'type': 'bias', 'bias': -0.5},
    'bias_late': {'type': 'bias', 'bias': 0.5}
}

# Monte Carlo parameters
N_MONTE_CARLO = 100
MONTE_CARLO_STD = 0.5

# Analysis parameters
TAU_MIN = 0.01
Q_VALUES = np.linspace(0.25, 5.0, 20)
SAMPLING_RATE = 200.0

# Signal columns
SIGNAL_COLUMN_MAP = {
    'acceleration': 'signal',
    'velocity': 'signal',
    'displacement': 'signal'
}

SIGNAL_COLUMN = SIGNAL_COLUMN_MAP[DATA_TYPE]

print(f"Configuration set for: {DATA_TYPE}")
print(f"Coda methods: {CODA_METHODS}")
print(f"Scenarios: {len(PERTURBATION_SCENARIOS)}")
print(f"Monte Carlo runs: {N_MONTE_CARLO}")

Configuration set for: acceleration
Coda methods: ['rautian', 'arias', 'envelope', 'median']
Scenarios: 5
Monte Carlo runs: 100


## 3. Load data

In [3]:
# Paths
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / 'data' / 'processed'
PICKS_DIR = DATA_DIR / f'03b_phase_identification_{PICKING_METHOD}' / DATA_TYPE
BASELINE_DIR = DATA_DIR / '04a_moment_scaling_spatial' / PICKING_METHOD / DATA_TYPE
OUTPUT_DIR = DATA_DIR / '05_sensitivity_analysis' / PICKING_METHOD / DATA_TYPE

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load signals
print(f"Loading data for: {DATA_TYPE}")
with open(PICKS_DIR / f'signals_{DATA_TYPE}_dict.pkl', 'rb') as f:
    signals_dict = pickle.load(f)
print(f"  Signals: {len(signals_dict)} stations")

# Load picks
df_full = pd.read_parquet(PICKS_DIR / f'df_full_{DATA_TYPE}_{PICKING_METHOD}.parquet')
print(f"  Picks: {len(df_full)} records")

# Load baseline results
baseline_results = {}
for coda_method in CODA_METHODS:
    baseline_file = BASELINE_DIR / coda_method / 'ensemble_spatial_summary.parquet'
    if baseline_file.exists():
        baseline_results[coda_method] = pd.read_parquet(baseline_file)
        print(f"  Baseline {coda_method}: loaded")

print("\nData loaded")

Loading data for: acceleration
  Signals: 21 stations
  Picks: 63 records
  Baseline rautian: loaded
  Baseline arias: loaded
  Baseline envelope: loaded
  Baseline median: loaded

Data loaded


## 4. Run sensitivity analysis

In [4]:
# Pack configuration
config = {
    'TAU_MIN': TAU_MIN,
    'Q_VALUES': Q_VALUES,
    'SAMPLING_RATE': SAMPLING_RATE,
    'SIGNAL_COLUMN': SIGNAL_COLUMN,
    'N_MONTE_CARLO': N_MONTE_CARLO,
    'MONTE_CARLO_STD': MONTE_CARLO_STD
}

# Run analysis
results = run_sensitivity_analysis(
    data_type=DATA_TYPE,
    signals_dict=signals_dict,
    df_full=df_full,
    baseline_results=baseline_results,
    coda_methods=CODA_METHODS,
    perturbation_scenarios=PERTURBATION_SCENARIOS,
    segment_function=segment_all_signals,
    analyze_function=analyze_all_windows,
    config=config,
    output_dir=OUTPUT_DIR,
    verbose=True
)


SENSITIVITY ANALYSIS: ACCELERATION
Coda methods: 4
Scenarios: 5 + Monte Carlo (100 runs)
Total combinations: 24


[ACCELERATION] Processing: rautian
--------------------------------------------------------------------------------
  [1/24] noise_small... Note: Converted pre_p_duration to 1000 samples @ 200Hz

SIGNAL SEGMENTATION SUMMARY
Successfully segmented: 63 signals from 21 stations
Skipped (no onset data): 0
Skipped (missing values): 0
Skipped (errors): 0

Coda detection method: rautian
Working unit: samples
Sampling rate: 200 Hz
Pre-event strategy: 1000

Pre-event window durations:
  Target: 1000 samples (5.00s)
  Actual mean: 5.00s (1000 samp)
  Actual range: [5.00, 5.00]s
ENSEMBLE SPATIAL SCALING ANALYSIS
tau_min: 0.010 s (fixed for all windows)
tau_max_fraction: None (use full window duration)
q_values: 20 values from 0.25 to 5.00
sampling_rate: 200.0 Hz

Processing window: PRE_EVENT
--------------------------------------------------------------------------------
Ensemble siz

## 5. Save results

In [ ]:
# Save complete results
results_file = OUTPUT_DIR / f'sensitivity_results_{DATA_TYPE}.pkl'
with open(results_file, 'wb') as f:
    pickle.dump(results, f)
print(f"Saved complete results: {results_file}")

# Create and save summary
df_summary = create_summary(
    results,
    data_type=DATA_TYPE,
    save_path=OUTPUT_DIR / f'sensitivity_summary_{DATA_TYPE}.csv'
)

print(f"Summary shape: {df_summary.shape}")
print(f"Saved summary: {OUTPUT_DIR / f'sensitivity_summary_{DATA_TYPE}.csv'}")

## 6. View summary

In [ ]:
# Display summary pivot table
print(f"\nSUMMARY: {DATA_TYPE.upper()}")
print("="*80)

summary_pivot = df_summary.pivot_table(
    index=['scenario', 'window'],
    columns='coda_method',
    values='rmse',
    aggfunc='mean'
)

print(summary_pivot)

# Top 10 worst cases
print(f"\n\nTOP 10 WORST CASES (highest RMSE)")
print("="*80)

worst_cases = df_summary.nlargest(10, 'rmse')[[
    'coda_method', 'scenario', 'window', 'rmse', 'mae', 'correlation'
]]
print(worst_cases.to_string(index=False))

## 7. Visualization

In [ ]:
# Heatmap: RMSE by scenario and window for each coda method
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, coda_method in enumerate(CODA_METHODS):
    
    ax = axes[idx]
    
    df_method = df_summary[df_summary['coda_method'] == coda_method]
    
    pivot = df_method.pivot_table(
        index='scenario',
        columns='window',
        values='rmse',
        aggfunc='mean'
    )
    
    im = ax.imshow(pivot.values, aspect='auto', cmap='YlOrRd')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=45, ha='right')
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_title(f'{coda_method}', fontsize=12, fontweight='bold')
    
    # Add values
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f'{val:.3f}', 
                       ha='center', va='center', fontsize=8)
    
    plt.colorbar(im, ax=ax, label='RMSE')

plt.suptitle(f'Sensitivity Analysis: {DATA_TYPE.upper()}', 
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()

fig_file = OUTPUT_DIR / f'sensitivity_heatmap_{DATA_TYPE}.png'
plt.savefig(fig_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"\nSaved figure: {fig_file}")